In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [9]:
def procesar_cruce_docentes_enoe(
    ruta_sdemt: str,
    ruta_coe1: str,
    ruta_salida: str = "../data/processed/docentes_enoe_reporte_limpio.csv",
    reporte_salida: str = "../data/processed/reporte_limpieza.txt",
) -> pd.DataFrame:
  reporte = []
  reporte.append("==================================================")
  reporte.append("      REPORTE DE LIMPIEZA - DOCENTES ENOE         ")
  reporte.append("==================================================")

  # 1. Carga de archivos
  if not os.path.exists(ruta_sdemt) or not os.path.exists(ruta_coe1):
    raise FileNotFoundError(
        "No se encontraron los archivos de la ENOE en la ruta especificada."
    )

  reporte.append(f"Archivos de origen: {ruta_sdemt} y {ruta_coe1}")

  # Definir llaves de cruce
  llaves_cruce = ["cve_ent", "con", "upm", "v_sel", "n_hog", "h_mud", "n_ren"]

  # Columnas requeridas
  cols_sdemt = llaves_cruce + [
      "eda",
      "sex",
      "niv_ins",
      "cs_p13_1",  
      "cs_p13_2",
      "anios_esc",
      "hrsocup",
      "ingocup",
      "ing_x_hrs",
      "pos_ocu",
      "seg_soc",
      "clase1",
      "fac_tri",
  ]
  cols_coe1 = llaves_cruce + ["p3", "p3a", "p4", "p4a","p3c1","p3c2","p3c3"]

  df_sdemt = pd.read_csv(
      ruta_sdemt, usecols=cols_sdemt, encoding="latin1", low_memory=False
  )
  df_coe1 = pd.read_csv(
      ruta_coe1, usecols=cols_coe1, encoding="latin1", low_memory=False
  )

  reporte.append(f"Filas iniciales SDEMT: {len(df_sdemt):,}")
  reporte.append(f"Filas iniciales COE1: {len(df_coe1):,}")

  # 2. Cruce de DataFrames
  df_unido = pd.merge(df_sdemt, df_coe1, on=llaves_cruce, how="inner")
  reporte.append(
      f"[7] Registros totales tras la unión (Inner Join): {len(df_unido):,}"
  )

  # 3. Filtrar personal docente (Códigos SINCO del grupo 23)
  df_docentes = df_unido[
      df_unido["p3"].astype(str).str.strip().str.startswith("23")
  ].copy()
  reporte.append(
      f"[8] Filtrado estricto de docentes (Grupo SINCO 23): {len(df_docentes):,}"
  )

  # 4. Normalización de nombres de columnas
  df_docentes.columns = (
      df_docentes.columns.astype(str)
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace(r"[^\w\s]", "", regex=True)
  )

  # 5. Estandarización de códigos especiales de INEGI ('\\N', espacios vacíos) a NaN
  df_docentes.replace([r"\\N", r"^\s*$", "NA", "NaN"], np.nan, regex=True, inplace=True)

  # 6. Conversión de tipos de datos principales a numéricos
  cols_numericas = [
    "eda", 
    "sex", 
    "anios_esc", 
    "hrsocup", 
    "ingocup", 
    "fac_tri", 
    "niv_ins", 
    "cs_p13_1",
  ]
  for col in cols_numericas:
    if col in df_docentes.columns:
      df_docentes[col] = pd.to_numeric(df_docentes[col], errors="coerce")

  # Filtrado de estructurales esenciales
  filas_antes_nulos = len(df_docentes)
  cols_esenciales = [
    "eda", 
    "sex", 
    "anios_esc", 
    "hrsocup", 
    "ingocup", 
    "fac_tri"
  ]
  df_docentes = df_docentes.dropna(subset=cols_esenciales).copy()
  reporte.append(
      f"[9] Filas eliminadas por valores nulos en variables esenciales: {filas_antes_nulos - len(df_docentes):,}"   
  )

  # Tratamiento para complementarias (p.ej. contrato p4 -> rellenar nulos con 99 'No especificado')
  if "p4" in df_docentes.columns:
    df_docentes["p4"] = df_docentes["p4"].fillna(99)


  # Depuración de valores lógicamente imposibles o absurdos
  filas_antes_rangos = len(df_docentes)
  df_docentes = df_docentes[
      (df_docentes["eda"] >= 18)
      & (df_docentes["hrsocup"] > 0)
      & (df_docentes["hrsocup"] <= 98)
      & (df_docentes["ingocup"] > 0)
  ].copy()
  reporte.append(
      f"[10] Filas depuradas por valores extraños o fuera de rango (edad < 18, hrs <= 0 o > 98, ingreso <= 0): {filas_antes_rangos - len(df_docentes):,}"
  )

  # 9. Resumen Final
  reporte.append("\n==================================================")
  reporte.append("                  RESUMEN FINAL                   ")
  reporte.append("==================================================")
  reporte.append(f"Filas finales limpias: {len(df_docentes)}")
  reporte.append(f"Columnas finales: {len(df_docentes.columns)}")
  reporte.append(f"Archivo limpio guardado en: {ruta_salida}")

  texto_reporte = "\n".join(reporte)
  print("\n" + texto_reporte + "\n")

  # Guardar reporte en texto
  if reporte_salida:
    with open(reporte_salida, "w", encoding="utf-8") as f:
      f.write(texto_reporte)

  # Guardar CSV final optimizado para MySQL y análisis
  df_docentes.to_csv(ruta_salida, index=False, encoding="utf-8", na_rep="\\N")
  return df_docentes

In [10]:
ruta_sdemt = "../data/raw/ENOE_SDEMT226.csv"
ruta_coe1 = "../data/raw/ENOE_COE1T226.csv"

df_docentes_limpio = procesar_cruce_docentes_enoe(
    ruta_sdemt = ruta_sdemt,
    ruta_coe1 = ruta_coe1,
    ruta_salida = "../data/processed/docentes_enoe_reporte_limpio.csv",
    reporte_salida = "../data/processed/reporte_limpieza.txt",
)
print("Vista previa del dataset generado:")
display(df_docentes_limpio.head())


      REPORTE DE LIMPIEZA - DOCENTES ENOE         
Archivos de origen: ../data/raw/ENOE_SDEMT226.csv y ../data/raw/ENOE_COE1T226.csv
Filas iniciales SDEMT: 417,128
Filas iniciales COE1: 344,709
[7] Registros totales tras la unión (Inner Join): 369,419
[8] Filtrado estricto de docentes (Grupo SINCO 23): 6,857
[9] Filas eliminadas por valores nulos en variables esenciales: 4
[10] Filas depuradas por valores extraños o fuera de rango (edad < 18, hrs <= 0 o > 98, ingreso <= 0): 3,401

                  RESUMEN FINAL                   
Filas finales limpias: 3452
Columnas finales: 27
Archivo limpio guardado en: ../data/processed/docentes_enoe_reporte_limpio.csv

Vista previa del dataset generado:


,cve_ent,con,upm,v_sel,n_hog,h_mud,n_ren,sex,eda,cs_p13_1,...,hrsocup,ingocup,ing_x_hrs,p3,p3a,p3c1,p3c2,p3c3,p4,p4a
15,9,40355,906125,52,1,0,2,2.0,48.0,7.0,...,50,32000,148.83721,2332,1,NaN,NaN,NaN,1,6112
153,21,40020,2106819,5,1,0,2,2.0,55.0,7.0,...,40,10000,58.13953,2322,1,NaN,NaN,NaN,1,6112
392,16,40357,1603947,5,1,0,1,1.0,32.0,7.0,...,40,12000,69.76744,2322,1,NaN,NaN,NaN,1,6112
408,15,41233,1530742,5,1,0,1,1.0,38.0,8.0,...,20,17200,200.00000,2321,1,NaN,NaN,NaN,1,6132
455,7,40067,709588,3,1,0,1,1.0,54.0,7.0,...,35,14000,93.02326,2322,1,NaN,NaN,NaN,1,6112
